# Query a Fabric Ontology through MCP

Connect directly to the Fabric IQ ontology MCP endpoint, inspect the tools it exposes, and call one of those tools with a natural-language question. The notebook signs you in to the Fabric tenant through the browser, and your account needs access to the Fabric workspace.

## Step 1: Load configuration and create an MCP session helper

Load the Fabric tenant and ontology MCP endpoint written to the project `.env` file during provisioning. The HTTP client follows redirects because Fabric routes MCP requests through its service endpoint.

In [ ]:
import os
from contextlib import asynccontextmanager

import httpx
from azure.identity import InteractiveBrowserCredential
from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

load_dotenv(override=True)

FABRIC_TENANT_ID = os.environ["FABRIC_TENANT_ID"]
FABRIC_ONTOLOGY_MCP_URL = os.environ["FABRIC_ONTOLOGY_MCP_URL"]
FABRIC_SCOPE = "https://api.fabric.microsoft.com/.default"

credential = InteractiveBrowserCredential(tenant_id=FABRIC_TENANT_ID)


@asynccontextmanager
async def open_ontology_session():
    token = credential.get_token(FABRIC_SCOPE)

    async with httpx.AsyncClient(
        headers={"Authorization": f"Bearer {token.token}"},
        follow_redirects=True,
        timeout=httpx.Timeout(30, read=300),
    ) as http_client:
        async with streamable_http_client(FABRIC_ONTOLOGY_MCP_URL, http_client=http_client,) as (read, write, _):
            async with ClientSession(read, write) as session:
                await session.initialize()
                yield session

print("Fabric ontology configuration loaded")

## Step 2: List the available tools

Initialize an authenticated MCP session and inspect every tool exposed by the Fabric ontology, including each tool's input schema.

In [ ]:
from rich.console import Console

console = Console()

async with open_ontology_session() as session:
    tools_response = await session.list_tools()

ontology_tools = tools_response.tools

for tool in ontology_tools:
    console.print(f"[bold cyan]Tool:[/] {tool.name}")
    if tool.description:
        console.print(f"[bold]Description:[/] {tool.description}")
    console.print("[bold]Input schema:[/]")
    console.print_json(data=tool.inputSchema)
    console.print()

## Step 3: Query the Fabric ontology

Call the ontology's `ask_ontology` tool with a natural-language business question. The tool answers from the ontology's bound lakehouse data and returns the answer with supporting evidence. Change `question` to explore other product, supplier, store, and inventory questions.

Answers typically take 30 to 60 seconds. If a question takes longer than about a minute, Fabric can return an empty response, which surfaces as `Error parsing JSON response` followed by a timeout. If that happens, re-run the cell.

In [ ]:
from datetime import timedelta

from rich.markdown import Markdown

question = "Which suppliers supply the Standard Outlet 15-Amp, and what are their lead times?"

async with open_ontology_session() as session:
    result = await session.call_tool(
        "ask_ontology",
        {"request": question},
        read_timeout_seconds=timedelta(seconds=120),
    )

answer = "".join(block.text for block in result.content if block.type == "text")
console.print(Markdown(answer))

## ✅ Mission complete

**What you built:**

* ✓ **Authenticated MCP session**: A direct connection to the ontology's MCP endpoint, signed in with your lab account.
* ✓ **Ontology answer**: `ask_ontology` answered a business question from the lakehouse data bound to the ontology.

➡️ Return to lab instructions.